In [1]:
import sys
from pathlib import Path

import torch

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from custom.cotsfa_original import (
    paper_contrastive_score,
    CoTSFAOriginalAlignmentLoss,
)

print("ROOT:", ROOT)

ROOT: C:\Users\cecil\Projects\Time-Series-Library


In [2]:
B = 8       # batch size
T = 96      # pred_len
Dz = 16     # latent dimension
Dy = 7      # target dimension
A = 5       # number of augmentations

torch.manual_seed(42)

# Original latent
z = torch.randn(B, T, Dz, requires_grad=True)

# Augmented latent: A augmentations per sample
z_aug = (
    z.detach().unsqueeze(0).repeat(A, 1, 1, 1)
    + 0.05 * torch.randn(A, B, T, Dz)
)
z_aug.requires_grad_(True)

# Original target
y = torch.randn(B, T, Dy)

# Input-only case: target unchanged
y_aug_input_only = y.unsqueeze(0).repeat(A, 1, 1, 1)

print("z:", z.shape)
print("z_aug:", z_aug.shape)
print("y:", y.shape)
print("y_aug_input_only:", y_aug_input_only.shape)

z: torch.Size([8, 96, 16])
z_aug: torch.Size([5, 8, 96, 16])
y: torch.Size([8, 96, 7])
y_aug_input_only: torch.Size([5, 8, 96, 7])


In [3]:
latent_scores = paper_contrastive_score(
    anchor=z,
    augmented=z_aug,
    temperature=1.0,
    include_original_negatives=True,
    include_same_sample_augmentations=True,
    normalize=False,
)

target_scores = paper_contrastive_score(
    anchor=y,
    augmented=y_aug_input_only,
    temperature=1.0,
    include_original_negatives=True,
    include_same_sample_augmentations=True,
    normalize=False,
)

print("latent_scores shape:", latent_scores.shape)
print("target_scores shape:", target_scores.shape)

print("latent_scores mean:", latent_scores.mean().item())
print("target_scores mean:", target_scores.mean().item())
print("latent_scores std:", latent_scores.std().item())
print("target_scores std:", target_scores.std().item())

latent_scores shape: torch.Size([5, 8, 96])
target_scores shape: torch.Size([5, 8, 96])
latent_scores mean: 1.6314871311187744
target_scores mean: 1.7553576231002808
latent_scores std: 0.1870204359292984
target_scores std: 0.27456265687942505


In [4]:
loss_fn = CoTSFAOriginalAlignmentLoss(
    temperature=1.0,
    include_original_negatives=True,
    include_same_sample_augmentations=True,
    normalize=False,
)

loss_align, latent_mean, target_mean = loss_fn(
    z=z,
    z_aug=z_aug,
    y=y,
    y_aug=y_aug_input_only,
)

print("Input-only case")
print("loss_align:", loss_align.item())
print("latent_score_mean:", latent_mean.item())
print("target_score_mean:", target_mean.item())

Input-only case
loss_align: 0.23235101997852325
latent_score_mean: 1.6314871311187744
target_score_mean: 1.7553576231002808


In [5]:
# Input-output case: target is changed
y_aug_changed = (
    y.unsqueeze(0).repeat(A, 1, 1, 1)
    + 0.5 * torch.randn(A, B, T, Dy)
)

loss_align_changed, latent_mean_changed, target_mean_changed = loss_fn(
    z=z,
    z_aug=z_aug,
    y=y,
    y_aug=y_aug_changed,
)

print("Input-output changed case")
print("loss_align:", loss_align_changed.item())
print("latent_score_mean:", latent_mean_changed.item())
print("target_score_mean:", target_mean_changed.item())

Input-output changed case
loss_align: 1.1520695686340332
latent_score_mean: 1.6314871311187744
target_score_mean: 2.37259578704834


In [6]:
details = loss_fn(
    z=z,
    z_aug=z_aug,
    y=y,
    y_aug=y_aug_changed,
    return_details=True,
)

print("loss_align:", details.loss_align.item())
print("latent_score_mean:", details.latent_score_mean.item())
print("target_score_mean:", details.target_score_mean.item())
print("latent_score_std:", details.latent_score_std.item())
print("target_score_std:", details.target_score_std.item())

loss_align: 1.1520695686340332
latent_score_mean: 1.6314871311187744
target_score_mean: 2.37259578704834
latent_score_std: 0.1870204359292984
target_score_std: 1.3163386583328247


In [7]:
loss = details.loss_align

loss.backward()

print("z grad is None:", z.grad is None)
print("z_aug grad is None:", z_aug.grad is None)

print("z grad mean abs:", z.grad.abs().mean().item())
print("z_aug grad mean abs:", z_aug.grad.abs().mean().item())

z grad is None: False
z_aug grad is None: False
z grad mean abs: 2.1565234419540502e-05
z_aug grad mean abs: 0.00015985452046152204
